# 00 — Generación del dataset

Este cuaderno es el punto de entrada del proyecto. A diferencia de los datasets típicos de imagen, en nuestro caso no hay un repositorio público con miles de fotografías etiquetadas: tenemos un catálogo de compuestos (~200) y partimos de cero. Cada imagen del dataset la generamos nosotros a partir del SMILES del compuesto.

La idea es la siguiente. Para los compuestos covalentes usamos RDKit para producir un dibujo 2D estandarizado de la molécula. Para los iónicos (sales, hidróxidos, etc.) RDKit no es buena opción porque su representación natural es la fórmula química, no una estructura de Lewis, así que los renderizamos como texto centrado sobre fondo blanco. A partir de cada imagen base aplicamos un pipeline de `Albumentations` para producir N variantes con rotaciones, ruido y deformaciones elásticas, intentando imitar la variabilidad de los dibujos a mano de un alumno de bachillerato.

El script `data/generate_dataset.py` automatiza todo este proceso, escribe las imágenes en `data/raw/<categoria>/<subcategoria>/<id>/` y genera `data/metadata.csv` con una fila por imagen ya con el split estratificado en train / val / test. Este notebook sirve para invocarlo cómodamente desde una UI y para verificar visualmente que lo que se genera tiene sentido antes de pasar al análisis y al entrenamiento.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
RAW_DATA_DIR  = ROOT / 'data' / 'raw'
MODELS_DIR    = ROOT / 'saved_models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Data found: {METADATA_PATH.exists() and METADATA_PATH.stat().st_size > 200}')

## Taxonomía

El catálogo está dividido en dos grandes categorías —Química Inorgánica y Química Orgánica— y cada una se subdivide en bloques temáticos siguiendo el esquema de bachillerato (óxidos, hidróxidos, oxoácidos, alcanos, alquenos, etc.). Toda esta taxonomía vive en `data/compounds.py` como una estructura de Python pura, sin ninguna dependencia interna del proyecto, y se consulta con la función `get_compounds()` que filtra por categoría, subcategoría o nivel de dificultad.

In [ ]:
from data.compounds import COMPOUNDS, TAXONOMY, get_compounds

print(f'Total compuestos: {len(COMPOUNDS)}')
for cat, cat_data in TAXONOMY.items():
    n_cat = sum(len(v["ids"]) for v in cat_data['subcategories'].values())
    print(f'\n[{cat}] — {cat_data["display"]} ({n_cat} compuestos)')
    for sub, sub_data in cat_data['subcategories'].items():
        print(f'   - {sub_data["display"]:30s} ({len(sub_data["ids"])} compuestos)')

## Lanzar la generación

El siguiente bloque envuelve el script en una pequeña interfaz con `ipywidgets` para no tener que recordar los argumentos de la CLI cada vez. Para un dataset de prueba bastan 5-10 imágenes por clase; para entrenar de verdad usamos 300, que es el valor que hemos utilizado en los notebooks 03 y 04. Generar 300 × 196 ≈ 58.000 imágenes lleva del orden de 15-25 minutos en un portátil normal y ocupa unos 4 GB en disco.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

category = widgets.RadioButtons(options=['all', 'inorganica', 'organica'], description='Categoría:')
n_per_class = widgets.IntSlider(value=20, min=5, max=300, step=5, description='N/clase:')
difficulty = widgets.SelectMultiple(options=['basico', 'intermedio', 'avanzado'],
                                     value=['basico', 'intermedio', 'avanzado'],
                                     description='Dificultad:')
out = widgets.Output()
btn = widgets.Button(description='Generar dataset', button_style='success')

def on_click(_):
    with out:
        clear_output()
        import subprocess
        cmd = [sys.executable, str(ROOT / 'data' / 'generate_dataset.py'),
               '--categories', category.value,
               '--n_per_class', str(n_per_class.value),
               '--difficulty', ','.join(difficulty.value),
               '--output_dir', str(RAW_DATA_DIR)]
        print('Ejecutando:', ' '.join(cmd))
        proc = subprocess.run(cmd, cwd=str(ROOT))
        print('\nReturn code:', proc.returncode)

btn.on_click(on_click)
display(widgets.VBox([category, n_per_class, difficulty, btn, out]))

## Inspección del CSV de metadatos

Una vez ejecutado el generador, todo el conocimiento del dataset queda volcado en `data/metadata.csv` (una fila por imagen, con su ruta, el `compound_id`, la subcategoría, la dificultad y el split). El resto de notebooks parten exclusivamente de este fichero, así que conviene asegurarse de que se ha escrito bien antes de seguir.

In [ ]:
import pandas as pd
if METADATA_PATH.exists() and METADATA_PATH.stat().st_size > 200:
    df = pd.read_csv(METADATA_PATH)
    print(f'metadata.csv: {len(df)} filas')
    display(df.head())
    print('\nDistribución por split:')
    print(df['split'].value_counts())
    print('\nDistribución por categoría:')
    print(df.groupby(['category', 'subcategory'])['compound_id'].nunique())
else:
    print('Aún no hay datos. Ejecuta la celda 2 (Generar dataset) primero.')

## Inspección visual

Pintamos 25 imágenes al azar (una por compuesto distinto) para comprobar que el render ha salido bien. En esta rejilla deberían verse tanto las estructuras 2D de RDKit como las fórmulas en texto de los iónicos. Si alguna imagen sale en blanco o con un símbolo extraño suele ser señal de un SMILES mal parseado.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

if METADATA_PATH.exists() and METADATA_PATH.stat().st_size > 200:
    df = pd.read_csv(METADATA_PATH)
    sample = df.groupby('compound_id').head(1).sample(min(25, df['compound_id'].nunique()), random_state=42)
    n = len(sample)
    cols = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.5))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else list(axes)
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(ROOT / row['filepath'])
        ax.imshow(img); ax.axis('off')
        ax.set_title(f"{row['formula']}\n({row['subcategory'][:15]})", fontsize=8)
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

## Variantes aumentadas

Otra comprobación importante: que las variantes generadas por `Albumentations` no estén destruyendo la información del compuesto. Aquí cogemos un compuesto y pintamos sus primeras 6 imágenes en disco. La imagen 0 es el render base (sin aumentación) y las siguientes son variantes ya distorsionadas. Si las distorsiones son demasiado agresivas (rotación >30°, ruido excesivo) tendremos un dataset imposible de aprender; si son demasiado suaves, el modelo no generalizará a dibujos a mano reales. Hemos calibrado los parámetros del pipeline en `src/augmentation.py` para mantener el equilibrio.

In [ ]:
if METADATA_PATH.exists() and METADATA_PATH.stat().st_size > 200:
    cid = df['compound_id'].iloc[0]
    rows = df[df['compound_id'] == cid].head(6)
    fig, axes = plt.subplots(1, len(rows), figsize=(len(rows)*2.2, 2.5))
    if len(rows) == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, rows.iterrows()):
        img = Image.open(ROOT / r['filepath'])
        ax.imshow(img); ax.axis('off')
    plt.suptitle(f'Variantes de {cid}'); plt.tight_layout(); plt.show()

## Notas sobre la generación

Algunas decisiones que hemos tomado y que conviene tener presentes al leer los siguientes notebooks:

- **Render dual (RDKit vs texto)**. Es la elección con más impacto en todo lo que viene después. El t-SNE del notebook 01 muestra los dos super-clusters que produce esta dualidad, y la matriz de confusión del notebook 03 confirma que los errores casi nunca cruzan la frontera entre los dos modos. Para una práctica de bachillerato esto es razonable: cuando un alumno escribe `NaCl` no dibuja una estructura, escribe la fórmula.
- **Aumentación en disco vs. en tiempo de entrenamiento**. Hemos optado por escribir las variantes a disco (300/clase). Es más rápido en cada época —cero overhead de transformación—, a costa de espacio. Para 196 clases × 300 imágenes son unos 4 GB; aceptable.
- **Split estratificado a nivel de imagen, no de compuesto**. Con un único render base por clase, el split estratificado a nivel de compuesto no tiene sentido (sólo tendríamos 1 imagen por clase). El split actual asegura que cada compuesto aparece en los tres splits, así que evaluamos *capacidad de generalización a variantes nuevas del mismo compuesto*, no a compuestos nuevos.